# Spatial GRN Integration of the Dentate Gyrus

## Step2:Gene Regulatory Network uisng SCENIC+:
## Tutorial 2b: pycisTarget

**Author:** Xuan Cao  
**Affiliation:** Icahn School of Medicine at Mount Sinai  
**Email:** xuan.cao@mssm.edu  

**Contributors:** Donghoon Lee, Panos Roussos  

**Date:** March 20, 2026  

---

### Overview
This notebook demonstrates a pipeline for spatial imputation and integration of gene regulatory networks (GRNs) in the dentate gyrus. Using Tangram, we map single-cell transcriptomic profiles onto spatial transcriptomics data to enable downstream GRN inference in a spatially resolved context.

### scenicplus version 
0.1.dev454+g9320635


### Contents
1. Define function 
2. Running pycisTarget  
  

# Imports

In [1]:
import numpy as np
import scanpy as sc

sc.set_figure_params(scanpy=True, dpi=100, dpi_save=150, fontsize=12)
sc.set_figure_params(dpi=100, color_map = 'viridis_r')
sc.settings.verbosity = 1
sc.logging.print_header()
from matplotlib.pyplot import rc_context
import pandas as pd
import pyranges as pr

import gc

import sys
import os
import pickle
from pathlib import Path
import glob

import warnings
warnings.simplefilter(action='ignore')

import pycistarget
pycistarget.__version__

/sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/sc/arion/pr

scanpy==1.8.2 anndata==0.8.0 umap==0.5.3 numpy==1.23.5 scipy==1.9.3 pandas==1.5.1 scikit-learn==1.3.2 statsmodels==0.13.5 python-igraph==0.10.2 pynndescent==0.5.11


'1.0.2.dev10+g5c35539'

# Set dirs

In [2]:
master_dir = 'SpatialGRN/Output'
cistopic_dir = f'{master_dir}/02a.pycisTopic'
cistarget_dir = f'{master_dir}/02b.pycisTarget'

tmp_dir = "tmp/"
log_dir=f'{cistopic_dir}/00.log'

if not os.path.exists(cistarget_dir):
    os.makedirs(cistarget_dir)
    
if not os.path.exists(tmp_dir):
    os.makedirs(tmp_dir)

if not os.path.exists(tmp_dir):
    os.makedirs(log_dir)

# Define function

In [3]:
def run_pycistarget_xuan(
    out_dir,
    tmp_dir,
    ostu_file,
    top3k_file,
    DARs_file,
    rankings_db,
    scores_db,
    motif_annotation,
    annot_file 
    
):
    import os
    import pickle

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    if not os.path.exists(tmp_dir):
        os.makedirs(tmp_dir)

    #Load data-----------------------------------
    region_bin_topics_otsu = pickle.load(open(ostu_file, 'rb'))
    region_bin_topics_top3k = pickle.load(open(top3k_file, 'rb'))
    markers_dict = pickle.load(open(DARs_file, 'rb'))

    # Convert to dictionary of pyranges objects.
    import pyranges as pr
    from pycistarget.utils import region_names_to_coordinates
    region_sets = {}
    region_sets['topics_otsu'] = {}
    region_sets['topics_top_3'] = {}
    region_sets['DARs'] = {}
    for topic in region_bin_topics_otsu.keys():
        regions = region_bin_topics_otsu[topic].index[region_bin_topics_otsu[topic].index.str.startswith('chr')] #only keep regions on known chromosomes
        region_sets['topics_otsu'][topic] = pr.PyRanges(region_names_to_coordinates(regions))
    for topic in region_bin_topics_top3k.keys():
        regions = region_bin_topics_top3k[topic].index[region_bin_topics_top3k[topic].index.str.startswith('chr')] #only keep regions on known chromosomes
        region_sets['topics_top_3'][topic] = pr.PyRanges(region_names_to_coordinates(regions))
    for DAR in markers_dict.keys():
        regions = markers_dict[DAR].index[markers_dict[DAR].index.str.startswith('chr')] #only keep regions on known chromosomes
        region_sets['DARs'][DAR] = pr.PyRanges(region_names_to_coordinates(regions))

    for key in region_sets.keys():
        print(f'{key}: {region_sets[key].keys()}')

    annot = pickle.load(open(annot_file, 'rb'))
    annot = annot.dropna(subset = ['Chromosome', 'Start'])
    # Check if chromosomes have chr
    check = region_sets[list(region_sets.keys())[0]]
    if not any(['chr' in c for c in check[list(check.keys())[0]].df['Chromosome']]):
        annot.Chromosome = annot.Chromosome.str.replace('chr', '')
    if not any(['chr' in x for x in annot.Chromosome]):
        annot.Chromosome = [f'chr{x}' for x in annot.Chromosome]

    # This function will run cistarget based and DEM based motif enrichment analysis 
    # first motif enrichment for all regions 
    # if run_without_promoters=TRUE. add analysis without promoter regions.
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    from scenicplus.wrappers.run_pycistarget import run_pycistarget
    run_pycistarget(region_sets = region_sets,
                    species = 'custom',
                    custom_annot = annot,
                    save_path = out_dir,
                    ctx_db_path = rankings_db,
                    dem_db_path = scores_db,
                    path_to_motif_annotations = motif_annotation,
                    run_without_promoters = True,
                    n_cpu = 40,
                    _temp_dir = tmp_dir,
                    annotation_version = 'v10nr_clust'
                    )

# Running pycisTarget

In [5]:
ostu_file = f'{cistopic_dir}/region_bin_topics_otsu.pkl'
top3k_file = f'{cistopic_dir}/region_bin_topics_top3k.pkl'
DARs_file = f'{cistopic_dir}/DAR_dict.pkl'
tmp_dir = 'tmp'

rankings_db = 'resources/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather'
scores_db = 'resources/hg38_screen_v10_clust.regions_vs_motifs.scores.feather'
motif_annotation = 'resources/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl'
annot_file = 'resources/hg38_annot.pkl'

run_pycistarget_xuan(
    cistarget_dir,
    tmp_dir,
    ostu_file,
    top3k_file,
    DARs_file,
    rankings_db,
    scores_db,
    motif_annotation,
    annot_file 
)

topics_otsu: dict_keys(['Topic1', 'Topic2', 'Topic3', 'Topic4', 'Topic5', 'Topic6', 'Topic7', 'Topic8', 'Topic9', 'Topic10', 'Topic11', 'Topic12', 'Topic13', 'Topic14', 'Topic15', 'Topic16'])
topics_top_3: dict_keys(['Topic1', 'Topic2', 'Topic3', 'Topic4', 'Topic5', 'Topic6', 'Topic7', 'Topic8', 'Topic9', 'Topic10', 'Topic11', 'Topic12', 'Topic13', 'Topic14', 'Topic15', 'Topic16'])
DARs: dict_keys(['Astro', 'EN', 'IN', 'Immune', 'OPC', 'Oligo', 'Vascular'])
2026-06-13 07:53:55,311 pycisTarget_wrapper INFO     /sc/arion/projects/CommonMind/xuan/03.SpatialGRN/02.pycisTarget folder already exists.
2026-06-13 07:53:55,350 pycisTarget_wrapper INFO     Loading cisTarget database for topics_otsu
2026-06-13 07:53:55,351 cisTarget    INFO     Reading cisTarget database
2026-06-13 07:55:26,175 pycisTarget_wrapper INFO     Running cisTarget for topics_otsu


2026-06-13 07:55:36,063	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


(ctx_internal_ray pid=2696293) 2026-06-13 07:55:51,268 cisTarget    INFO     Running cisTarget for Topic1 which has 26582 regions
(ctx_internal_ray pid=2696276) 2026-06-13 07:55:58,991 cisTarget    INFO     Running cisTarget for Topic11 which has 22376 regions [repeated 10x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/ray-logging.html#log-deduplication for more options.)
(ctx_internal_ray pid=2696300) 2026-06-13 07:56:04,949 cisTarget    INFO     Running cisTarget for Topic14 which has 26436 regions [repeated 3x across cluster]
(ctx_internal_ray pid=2696293) 2026-06-13 07:56:26,786 cisTarget    INFO     Annotating motifs for Topic1
(ctx_internal_ray pid=2696290) 2026-06-13 07:56:09,065 cisTarget    INFO     Running cisTarget for Topic16 which has 22367 regions [repeated 2x across cluster]
(ctx_internal_ray pid=2696293) 2026-06-13 07:56:29,493 cisTarget    INFO     Getting cis

2026-06-13 07:57:27,514	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


(ctx_internal_ray pid=2700082) 2026-06-13 07:57:33,842 cisTarget    INFO     Running cisTarget for Topic1 which has 24247 regions
(ctx_internal_ray pid=2700069) 2026-06-13 07:57:40,925 cisTarget    INFO     Running cisTarget for Topic6 which has 18641 regions [repeated 5x across cluster]
(ctx_internal_ray pid=2700073) 2026-06-13 07:57:48,223 cisTarget    INFO     Running cisTarget for Topic10 which has 16695 regions [repeated 4x across cluster]
(ctx_internal_ray pid=2700067) 2026-06-13 07:57:55,468 cisTarget    INFO     Running cisTarget for Topic14 which has 19631 regions [repeated 4x across cluster]
(ctx_internal_ray pid=2700080) 2026-06-13 07:57:57,030 cisTarget    INFO     Annotating motifs for Topic3
(ctx_internal_ray pid=2700080) 2026-06-13 07:58:00,559 cisTarget    INFO     Getting cistromes for Topic3
(ctx_internal_ray pid=2700083) 2026-06-13 07:57:58,879 cisTarget    INFO     Running cisTarget for Topic16 which has 10595 regions [repeated 2x across cluster]
(ctx_internal_ray p

...

(ctx_internal_ray pid=2723444) 2026-06-13 08:15:29,487 cisTarget    INFO     Running cisTarget for Astro which has 34954 regions
(ctx_internal_ray pid=2723417) 2026-06-13 08:15:29,487 cisTarget    INFO     Running cisTarget for EN which has 11769 regions
(ctx_internal_ray pid=2723436) 2026-06-13 08:15:29,491 cisTarget    INFO     Running cisTarget for IN which has 11134 regions
(ctx_internal_ray pid=2723450) 2026-06-13 08:15:29,486 cisTarget    INFO     Running cisTarget for Immune which has 10117 regions
(ctx_internal_ray pid=2723453) 2026-06-13 08:15:29,486 cisTarget    INFO     Running cisTarget for OPC which has 21088 regions
(ctx_internal_ray pid=2723433) 2026-06-13 08:15:29,504 cisTarget    INFO     Running cisTarget for Oligo which has 22953 regions
(ctx_internal_ray pid=2723431) 2026-06-13 08:15:29,506 cisTarget    INFO     Running cisTarget for Vascular which has 7346 regions
(ctx_internal_ray pid=2723436) 2026-06-13 08:15:46,302 cisTarget    INFO     Annotating motifs for IN


2026-06-13 08:17:07,590	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


(DEM_internal_ray pid=2726837) 2026-06-13 08:17:14,632 DEM          INFO     Computing DEM for Vascular
(DEM_internal_ray pid=2726835) 2026-06-13 08:17:15,521 DEM          INFO     Computing DEM for Immune
(DEM_internal_ray pid=2726838) 2026-06-13 08:17:15,742 DEM          INFO     Computing DEM for IN
(DEM_internal_ray pid=2726840) 2026-06-13 08:17:15,830 DEM          INFO     Computing DEM for EN
(DEM_internal_ray pid=2726834) 2026-06-13 08:17:18,305 DEM          INFO     Computing DEM for Oligo
(DEM_internal_ray pid=2726839) 2026-06-13 08:17:18,486 DEM          INFO     Computing DEM for OPC
(DEM_internal_ray pid=2726833) 2026-06-13 08:17:20,387 DEM          INFO     Computing DEM for Astro
2026-06-13 08:19:29,569 DEM          INFO     Forming cistromes
2026-06-13 08:19:31,714 DEM          INFO     Done!
2026-06-13 08:19:32,815 pycisTarget_wrapper INFO     Created folder : /sc/arion/projects/CommonMind/xuan/03.SpatialGRN/02.pycisTarget/DEM_DARs_All
2026-06-13 08:19:32,892 pycisTarge

2026-06-13 08:19:48,340	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


(DEM_internal_ray pid=2728127) 2026-06-13 08:19:55,152 DEM          INFO     Computing DEM for Vascular
(DEM_internal_ray pid=2728132) 2026-06-13 08:19:55,713 DEM          INFO     Computing DEM for Immune
(DEM_internal_ray pid=2728124) 2026-06-13 08:19:55,900 DEM          INFO     Computing DEM for IN
(DEM_internal_ray pid=2728128) 2026-06-13 08:19:56,081 DEM          INFO     Computing DEM for EN
(DEM_internal_ray pid=2728131) 2026-06-13 08:19:58,031 DEM          INFO     Computing DEM for OPC
(DEM_internal_ray pid=2728126) 2026-06-13 08:19:58,426 DEM          INFO     Computing DEM for Oligo
(DEM_internal_ray pid=2728130) 2026-06-13 08:20:00,863 DEM          INFO     Computing DEM for Astro
2026-06-13 08:20:47,121 DEM          INFO     Forming cistromes
2026-06-13 08:20:49,039 DEM          INFO     Done!
2026-06-13 08:20:50,152 pycisTarget_wrapper INFO     Created folder : /sc/arion/projects/CommonMind/xuan/03.SpatialGRN/02.pycisTarget/DEM_DARs_No_promoters
2026-06-13 08:20:50,229 p